# Inference problems

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

## Q7

In [2]:
df = pd.read_csv('./data/ames_prices.csv')

df['Central.Air'].value_counts(dropna=False)

Central.Air
Y    2734
N     196
Name: count, dtype: int64

In [3]:
central_air = (df['Central.Air'] == 'Y').astype(int)

p_hat = central_air.mean()
n = len(central_air)

print('Sample size:', n)
print('Estimated proportion:', p_hat)

Sample size: 2930
Estimated proportion: 0.9331058020477816


In [4]:
seed = 5030
rng = np.random.default_rng(seed)

R = 5000
results = []

for t in range(R):
    idx = rng.integers(0, n, size=n)
    p_hat_t = central_air.iloc[idx].mean()
    results.append(p_hat_t)

results = np.array(results)

In [5]:
standard_error = np.std(results, ddof=1)

print('Bootstrap standard error:', standard_error)

Bootstrap standard error: 0.004634021837179411


In [6]:
ci90 = (
    p_hat - 1.645 * standard_error,
    p_hat + 1.645 * standard_error
)

ci95 = (
    p_hat - 1.96 * standard_error,
    p_hat + 1.96 * standard_error
)

ci99 = (
    p_hat - 2.576 * standard_error,
    p_hat + 2.576 * standard_error
)

print('90% CI:', ci90)
print('95% CI:', ci95)
print('99% CI:', ci99)

90% CI: (0.9254828361256214, 0.9407287679699418)
95% CI: (0.9240231192469099, 0.9421884848486533)
99% CI: (0.9211685617952075, 0.9450430423003557)


In [7]:
p0 = 0.90

z = (p_hat - p0) / standard_error
p_value = 2 * norm.cdf(-abs(z))

print('z-statistic:', z)
print('Two-sided p-value:', p_value)

z-statistic: 7.144075537618113
Two-sided p-value: 9.060356980019481e-13


We used R=5,000 with seed=5030. The estimated proportion is 0.9331, with bootstrap SE = 0.004634.

| Confidence level | Confidence Interval |
|---|---|
| 90% | (0.92548, 0.94073) |
| 95% | (0.92402, 0.94219) |
| 99% | (0.92117, 0.94504) |

For $H_0:p=0.90$ vs. $H_1:p\ne0.90$, $z\approx7.1441$ and $p\approx9.06\times10^{-13}$. Since $|z|>2.576$ and $p<0.01$, we reject the null hypothesis at all three significance levels. All three confidence intervals exclude 0.90, agreeing with the every test decision.

Approximately 93.31% of homes have central air, 3.31 percentage points above 90%. The 95% CI is 92.40% to 94.22%. Under the sampling assumptions, these results provide strong evidence that the population proportion exceeds 90%. But this is a descriptive estimate, not a causal conclusion. The p-value is far below all three significance thresholds, so increasing $R$ is unnecessary.

## Q8

In [8]:
housing = df[['area', 'price']].dropna()

area = np.arcsinh(housing['area'])
price = np.arcsinh(housing['price'])

rho_hat = np.corrcoef(area, price)[0, 1]
n = len(housing)

print('Sample size:', n)
print('Estimated correlation:', rho_hat)

Sample size: 2930
Estimated correlation: 0.7231661530091358


In [9]:
seed = 5030
rng = np.random.default_rng(seed)

R = 2000
results = []

for t in range(R):
    idx = rng.integers(0, n, size=n)
    rho_hat_t = np.corrcoef(
        area.iloc[idx],
        price.iloc[idx]
    )[0, 1]
    results.append(rho_hat_t)

results = np.array(results)

In [10]:
standard_error = np.std(results, ddof=1)

ci90 = (
    rho_hat - 1.645 * standard_error,
    rho_hat + 1.645 * standard_error
)

ci95 = (
    rho_hat - 1.96 * standard_error,
    rho_hat + 1.96 * standard_error
)

ci99 = (
    rho_hat - 2.576 * standard_error,
    rho_hat + 2.576 * standard_error
)

print('Bootstrap standard error:', standard_error)
print('90% CI:', ci90)
print('95% CI:', ci95)
print('99% CI:', ci99)

Bootstrap standard error: 0.010015137425957605
90% CI: (0.7066912519434355, 0.739641054074836)
95% CI: (0.7035364836542589, 0.7427958223640126)
99% CI: (0.697367158999869, 0.7489651470184026)


In [11]:
rho0 = 0

z = (rho_hat - rho0) / standard_error
p_value = 2 * norm.cdf(-abs(z))

print('z-statistic:', z)
print('Two-sided p-value:', p_value)

z-statistic: 72.2073120169881
Two-sided p-value: 0.0


We used 2,000 paired rows bootstrap samples with seed 5030. The estimated Pearson correlation between arcsinh(area) and arcsinh(price) was 0.7232, with bootstrap SE = 0.010015.

| Confidence level | Normal CI |
|---|---|
| 90% | (0.70669, 0.73964) |
| 95% | (0.70354, 0.74280) |
| 99% | (0.69737, 0.74897) |

For $H_0:\rho=0$ vs. $H_1:\rho\ne0$, $z=72.2073$. The two-sided p-value is extremely small ($p<0.001$), showed as 0.0 due to numerical underflow. Since $|z|>2.576$ and $p<0.01$, we reject $H_0$ at the 10%, 5%, and 1% levels. All three CIs exclude zero, agree with these decisions. Increasing R was unnecessary because the result is far from all decision thresholds.

The transformed variables show a strong positive linear relationship, larger homes tend to have higher prices. This does not establish causation. Resampling area and price separately would break the observed pairs and destroy their dependence, producing an invalid bootstrap SE for the correlation.

## Q9

In [12]:
metabric = pd.read_csv('./data/metabric.csv')

survival = metabric['Overall Survival (Months)'].dropna()

mean_hat = survival.mean()
n = len(survival)

print('Sample size:', n)
print('Estimated mean (months):', mean_hat)
print('Estimated mean (years):', mean_hat / 12)

Sample size: 1343
Estimated mean (months): 129.09429138744105
Estimated mean (years): 10.757857615620088


In [13]:
seed = 5030
rng = np.random.default_rng(seed)

R = 5000
results = []

for t in range(R):
    idx = rng.integers(0, n, size=n)
    mean_hat_t = survival.iloc[idx].mean()
    results.append(mean_hat_t)

results = np.array(results)

standard_error = np.std(results, ddof=1)

print('Bootstrap standard error (months):', standard_error)

Bootstrap standard error (months): 2.1685356757791263


In [14]:
ci90 = (
    mean_hat - 1.645 * standard_error,
    mean_hat + 1.645 * standard_error
)

ci95 = (
    mean_hat - 1.96 * standard_error,
    mean_hat + 1.96 * standard_error
)

ci99 = (
    mean_hat - 2.576 * standard_error,
    mean_hat + 2.576 * standard_error
)

print('90% CI (months):', ci90)
print('95% CI (months):', ci95)
print('99% CI (months):', ci99)

print('90% CI (years):', np.array(ci90) / 12)
print('95% CI (years):', np.array(ci95) / 12)
print('99% CI (years):', np.array(ci99) / 12)

90% CI (months): (125.52705020078439, 132.66153257409772)
95% CI (months): (124.84396146291395, 133.34462131196813)
99% CI (months): (123.50814348663401, 134.68043928824807)
90% CI (years): [10.46058752 11.05512771]
95% CI (years): [10.40366346 11.11205178]
99% CI (years): [10.29234529 11.22336994]


In [15]:
mu0 = 120

z = (mean_hat - mu0) / standard_error
p_value = 2 * norm.cdf(-abs(z))

print('z-statistic:', z)
print('Two-sided p-value:', p_value)

z-statistic: 4.193747646864785
Two-sided p-value: 2.743832178344953e-05


We used 1,343 patients with observed recorded times, including both living and deceased patients, the estimated mean was 129.09 months (10.76 years). We used 5,000 bootstrap samples with seed 5030, bootstrap SE is approximately 2.1685 months.

| Confidence level | CI in months | CI in years |
|---|---|---|
| 90% | (125.53, 132.66) | (10.46, 11.06) |
| 95% | (124.84, 133.34) | (10.40, 11.11) |
| 99% | (123.51, 134.68) | (10.29, 11.22) |

For $H_0:\mu=120$ vs. $H_1:\mu\ne120$, $z=4.1937$ and $p\approx2.74\times10^{-5}$. Since $|z|>2.576$ and $p<0.01$, we reject $H_0$ at the 10%, 5%, and 1% significance levels. All three CIs exclude 120 months (10 years), agreeing with these decisions. No increase in R was needed.

The results support a population mean recorded time above 120 months. This variable measures time to death or last follow-up. Living patients' eventual death times are right-censored, so these intervals describe mean recorded time, not unrestricted mean lifetime.

## Q10

In [16]:
cancer = metabric[
    ['Chemotherapy', 'Overall Survival Status']
].dropna().copy()

cancer['deceased'] = (
    cancer['Overall Survival Status'] == '1:DECEASED'
).astype(int)

yes = cancer.loc[cancer['Chemotherapy'] == 'YES', 'deceased']
no = cancer.loc[cancer['Chemotherapy'] == 'NO', 'deceased']

delta_hat = yes.mean() - no.mean()

print('YES sample size:', len(yes))
print('NO sample size:', len(no))
print('YES deceased proportion:', yes.mean())
print('NO deceased proportion:', no.mean())
print('Estimated difference (YES minus NO):', delta_hat)

YES sample size: 286
NO sample size: 1057
YES deceased proportion: 0.5209790209790209
NO deceased proportion: 0.5856196783349101
Estimated difference (YES minus NO): -0.0646406573558892


In [17]:
seed = 5030
rng = np.random.default_rng(seed)

R = 100_000
results = []

for t in range(R):
    idx_yes = rng.integers(0, len(yes), size=len(yes))
    idx_no = rng.integers(0, len(no), size=len(no))

    delta_hat_t = (
        yes.iloc[idx_yes].mean() - no.iloc[idx_no].mean()
    )
    results.append(delta_hat_t)

results = np.array(results)

standard_error = np.std(results, ddof=1)

print('Bootstrap standard error:', standard_error)

Bootstrap standard error: 0.033281811273148935


In [18]:
ci90 = (
    delta_hat - 1.645 * standard_error,
    delta_hat + 1.645 * standard_error
)

ci95 = (
    delta_hat - 1.96 * standard_error,
    delta_hat + 1.96 * standard_error
)

ci99 = (
    delta_hat - 2.576 * standard_error,
    delta_hat + 2.576 * standard_error
)

z = delta_hat / standard_error
p_value = 2 * norm.cdf(-abs(z))

print('90% CI:', ci90)
print('95% CI:', ci95)
print('99% CI:', ci99)
print('z-statistic:', z)
print('Two-sided p-value:', p_value)

90% CI: (-0.11938923690021919, -0.009892077811559201)
95% CI: (-0.12987300745126112, 0.000591692739482716)
99% CI: (-0.15037460319552087, 0.021093288483742464)
z-statistic: -1.9422217386359593
Two-sided p-value: 0.05211026251201294


The deceased proportions were 52.10% in the chemotherapy group (n = 286) and 58.56% in the non-chemotherapy group (n = 1,057). The estimated difference (YES minus NO) was −0.06464. We used 100,000 independent within group bootstrap samples with seed=5030, the estimated SE was 0.03328.

| Confidence level | Normal CI |
|---|---|
| 90% | (-0.11939, -0.00989) |
| 95% | (-0.12987, 0.00059) |
| 99% | (-0.15037, 0.02109) |

For $H_0:\Delta=0$ vs. $H_1:\Delta\ne0$, $z=-1.9422$ and $p=0.05211$. 

- 90% confidence / 0.1 significance: $|z|=1.9422>1.645$, $p=0.05211<0.10$, and the 90% CI excludes zero. Reject $H_0$.

- 95% confidence / 0.05 significance: $|z|=1.9422<1.96$, $p=0.05211>0.05$, and the 95% CI contains zero. Fail to reject $H_0$.

- 99% confidence / 0.01 significance: $|z|=1.9422<2.576$, $p=0.05211>0.01$, and the 99% CI contains zero. Fail to reject $H_0$.

All three methods give consistent decisions at each level. We increased R to reduce Monte Carlo error, the result remains close to the 5% threshold, so there are some remaining simulation uncertainty.

The chemotherapy group had a lower observed deceased proportion, but the difference was not statistically significant at 5% and 1%. Possible confounders include age at diagnosis and tumor size. These observational results do not establish a causal effect of chemotherapy on mortality.

## Q11

In [19]:
nhanes = pd.read_csv('./data/nhanes_data_17_18.csv', low_memory=False)

sleep = nhanes.loc[
    nhanes['AgeInYearsAtScreening'] >= 18,
    'SleepHoursWeekdaysOrWorkdays'
].dropna()

mean_hat = sleep.mean()
n = len(sleep)

print('Sample size:', n)
print('Estimated mean sleep (hours):', mean_hat)
print('Difference from 8 hours:', mean_hat - 8)
print('Difference from 8 hours (minutes):', (mean_hat - 8) * 60)

Sample size: 5489
Estimated mean sleep (hours): 7.62124248496994
Difference from 8 hours: -0.3787575150300597
Difference from 8 hours (minutes): -22.725450901803583


In [22]:
seed = 5030
rng = np.random.default_rng(seed)

R = 5000
results = []

for t in range(R):
    idx = rng.integers(0, n, size=n)
    mean_hat_t = sleep.iloc[idx].mean()
    results.append(mean_hat_t)

results = np.array(results)

standard_error = np.std(results, ddof=1)
print('Bootstrap standard error:', standard_error)

Bootstrap standard error: 0.02281860253240433


In [21]:
ci90 = (
    mean_hat - 1.645 * standard_error,
    mean_hat + 1.645 * standard_error
)

ci95 = (
    mean_hat - 1.96 * standard_error,
    mean_hat + 1.96 * standard_error
)

ci99 = (
    mean_hat - 2.576 * standard_error,
    mean_hat + 2.576 * standard_error
)

z = (mean_hat - 8) / standard_error
p_value = 2 * norm.cdf(-abs(z))

print('90% CI (hours):', ci90)
print('95% CI (hours):', ci95)
print('99% CI (hours):', ci99)
print('z-statistic:', z)
print('Two-sided p-value:', p_value)

90% CI (hours): (7.583705883804135, 7.658779086135746)
95% CI (hours): (7.576518024006428, 7.6659669459334525)
99% CI (hours): (7.562461764846467, 7.680023205093414)
z-statistic: -16.598628881509825
Two-sided p-value: 7.1301104971668764e-62


Among 5,489 adults with observed weekday sleep, the estimated mean is 7.6212 hours, approximately 22.73 minutes below 8 hours. Using 5,000 bootstrap samples with seed 5030, the estimated SE was 0.02282 hours.

| Confidence level | Normal CI (hours) |
|---|---|
| 90% | (7.5837, 7.6588) |
| 95% | (7.5765, 7.6660) |
| 99% | (7.5625, 7.6800) |

For $H_0:\mu=8$ versus $H_1:\mu\ne8$, $|z|=16.5986>2.576$ and $p\approx7.13\times10^{-62}<0.01$. We reject $H_0$ at the 10%, 5%, and 1% levels. All three CIs exclude 8, consistent with these decisions. Increasing R was unnecessary.

The mean is statistically significantly below 8 hours. Practical importance depends on context and cannot be determined from the p-value alone.